Gauntlet #3: Clean the Titanic Dataset from Scratch


In [1]:
import pandas as pd
import numpy as np

# Load fresh Titanic dataset
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# Task 1: Initial Inspection
print("=== Shape ===")
print(df.shape)

print("\n=== Info ===")
df.info()

print("\n=== Missing Values per Column ===")
print(df.isna().sum())

# Identify columns with missing values
missing_cols = df.columns[df.isna().any()].tolist()
print(f"\nColumns with missing values: {missing_cols}")

=== Shape ===
(891, 12)

=== Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

=== Missing Values per Column ===
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0


In [2]:
# Task 2: Drop Cabin column
df = df.drop('Cabin', axis=1)
print("Cabin dropped. Remaining columns:", df.columns.tolist())

# Task 3: Fill missing Age with median
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)
print(f"Missing Age after fill: {df['Age'].isna().sum()}")

# Task 4: Fill missing Embarked with mode
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(mode_embarked)
print(f"Missing Embarked after fill: {df['Embarked'].isna().sum()}")

Cabin dropped. Remaining columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked']
Missing Age after fill: 0
Missing Embarked after fill: 0


In [3]:
# Task 5: Data type conversions
df['Survived'] = df['Survived'].astype(bool)
df['Pclass'] = df['Pclass'].astype('category')
df['Sex'] = df['Sex'].astype('category')

print("Updated dtypes:")
print(df.dtypes)

Updated dtypes:
PassengerId       int64
Survived           bool
Pclass         category
Name             object
Sex            category
Age             float64
SibSp             int64
Parch             int64
Ticket           object
Fare            float64
Embarked         object
dtype: object


In [4]:
# Task 6: Extract Title from Name
df['Title'] = df['Name'].str.extract(r'([A-Za-z]+)\.')
df['Title'] = df['Title'].str.lower()

# Define rare titles to be grouped as 'Other'
rare_titles = ['mlle', 'ms', 'mme', 'lady', 'sir', 'countess', 
               'jonkheer', 'don', 'rev', 'dr', 'major', 'col', 'capt']

df['Title'] = df['Title'].replace(rare_titles, 'Other')

print("Title value counts after cleaning:")
print(df['Title'].value_counts())

Title value counts after cleaning:
Title
mr        517
miss      182
mrs       125
master     40
Other      27
Name: count, dtype: int64


In [5]:
# Task 7: Rename columns
df = df.rename(columns={
    'PassengerId': 'passenger_id',
    'Pclass': 'pclass',
    'SibSp': 'siblings_spouses',
    'Parch': 'parents_children',
    'Fare': 'fare',
    'Embarked': 'embarked'
})

print("Renamed columns:", df.columns.tolist())

Renamed columns: ['passenger_id', 'Survived', 'pclass', 'Name', 'Sex', 'Age', 'siblings_spouses', 'parents_children', 'Ticket', 'fare', 'embarked', 'Title']


In [6]:
# Task 8: family_size, is_alone, age_group
df['family_size'] = df['siblings_spouses'] + df['parents_children'] + 1
df['is_alone'] = df['family_size'] == 1

bins = [0, 12, 18, 35, 60, 100]
labels = ['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
df['age_group'] = pd.cut(df['Age'], bins=bins, labels=labels)

print(df[['family_size', 'is_alone', 'age_group']].head())

   family_size  is_alone    age_group
0            2     False  Young Adult
1            2     False        Adult
2            1      True  Young Adult
3            2     False  Young Adult
4            1      True  Young Adult


In [ ]:
# Task 9: Map Title strings to numeric codes
title_mapping = {
    'mr': 0,
    'mrs': 1,
    'miss': 2,
    'master': 3,
    'other': 4
}
df['Title'] = df['Title'].map(title_mapping)

print("Title numeric codes:")
print(df['Title'].value_counts().sort_index())

In [7]:
# Task 10: Dummies for embarked, drop first, and concatenate
embarked_dummies = pd.get_dummies(df['embarked'], prefix='emb', drop_first=True)
df = pd.concat([df, embarked_dummies], axis=1)

# Drop the original embarked column
df = df.drop('embarked', axis=1)

print("Final DataFrame shape:", df.shape)
print("Final columns:", df.columns.tolist())

Final DataFrame shape: (891, 16)
Final columns: ['passenger_id', 'Survived', 'pclass', 'Name', 'Sex', 'Age', 'siblings_spouses', 'parents_children', 'Ticket', 'fare', 'Title', 'family_size', 'is_alone', 'age_group', 'emb_Q', 'emb_S']
